In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("ecommerce_raw.csv")
print("Shape:", df.shape)
df.head()

Shape: (520, 12)


,order_id,customer_id,customer_name,gender,order_date,product,category,quantity,unit_price,status,payment_method,region
0,ORD-1275,CUST-133,U Kyaw,Male,01-03-2023,Keyboard,Accessories,2,85.03,completed,PAYPAL,north
1,ORD-1093,CUST-255,Daw Mya,M,2023-12-13,Headphones,Audio,5,147.87,Completed,credit card,East
2,ORD-1006,CUST-157,Daw Su,male,2023/07/14,Webcam,Accessories,2,-89.00,COMPLETED,credit card,West
3,ORD-1167,CUST-129,Ko Aung,MALE,21-10-2023,Mouse,Accessories,2,43.30,Pending,Cash,East
4,ORD-1090,CUST-274,Ma Nwe,Female,06/03/2023,Mouse,Accessories,3,48.79,Pending,Credit Card,East


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        520 non-null    object 
 1   customer_id     520 non-null    object 
 2   customer_name   488 non-null    object 
 3   gender          496 non-null    object 
 4   order_date      520 non-null    object 
 5   product         520 non-null    object 
 6   category        520 non-null    object 
 7   quantity        520 non-null    int64  
 8   unit_price      489 non-null    float64
 9   status          498 non-null    object 
 10  payment_method  496 non-null    object 
 11  region          490 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 48.9+ KB


In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})
print(missing_df[missing_df["Missing Count"] > 0])

                Missing Count  Missing %
customer_name              32       6.15
gender                     24       4.62
unit_price                 31       5.96
status                     22       4.23
payment_method             24       4.62
region                     30       5.77


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 20


In [ ]:
for col in ["gender", "status", "payment_method", "region"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- gender ---
gender
F         67
M         65
MALE      64
Male      64
Female    61
female    59
male      58
FEMALE    58
Name: count, dtype: int64

--- status ---
status
COMPLETED    93
completed    79
pending      74
Cancelled    69
Completed    66
Pending      65
Returned     52
Name: count, dtype: int64

--- payment_method ---
payment_method
credit card    100
Debit Card      73
Cash            69
paypal          69
PAYPAL          67
Credit Card     60
PayPal          58
Name: count, dtype: int64

--- region ---
region
West     130
north     67
SOUTH     67
East      60
North     59
South     54
East      53
Name: count, dtype: int64


In [ ]:
print(df["unit_price"].describe())
print("\nNegative prices:", (df["unit_price"] < 0).sum())
print("Extreme prices (>5000):", (df["unit_price"] > 5000).sum())

count      489.000000
mean      1024.383967
std       6856.806416
min       -999.000000
25%         50.090000
50%         88.260000
75%        405.590000
max      99900.000000
Name: unit_price, dtype: float64

Negative prices: 9
Extreme prices (>5000): 11


In [ ]:
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (500, 12)


In [ ]:
# gender → Male / Female အဖြစ် သတ်မှတ်
def clean_gender(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().upper()
    if val in ["M", "MALE"]:
        return "Male"
    elif val in ["F", "FEMALE"]:
        return "Female"
    return np.nan

df["gender"] = df["gender"].apply(clean_gender)

# status, payment_method, region → title case + strip
df["status"] = df["status"].str.strip().str.title()
df["payment_method"] = df["payment_method"].str.strip().str.title()
df["region"] = df["region"].str.strip().str.title()

print("Gender unique:", df["gender"].unique())
print("Status unique:", df["status"].unique())
print("Payment unique:", df["payment_method"].unique())
print("Region unique:", df["region"].unique())

Gender unique: ['Male' 'Female' nan]
Status unique: ['Completed' 'Pending' 'Returned' 'Cancelled' nan]
Payment unique: ['Paypal' 'Credit Card' 'Cash' 'Debit Card' nan]
Region unique: ['North' 'East' 'West' nan 'South']


In [ ]:
# Negative prices → NaN အဖြစ် မှတ်တာ
df.loc[df["unit_price"] < 0, "unit_price"] = np.nan

# Extreme outliers (>5000) → NaN အဖြစ် မှတ်တာ
df.loc[df["unit_price"] > 5000, "unit_price"] = np.nan

print("Price stats after fix:")
print(df["unit_price"].describe())

Price stats after fix:
count     453.000000
mean      286.908366
std       399.650508
min        29.700000
25%        51.540000
50%        88.340000
75%       402.530000
max      3900.000000
Name: unit_price, dtype: float64


In [ ]:
# unit_price → product အလိုက် median နဲ့ ဖြည့်တာ
df["unit_price"] = df.groupby("product")["unit_price"].transform(
    lambda x: x.fillna(x.median())
)

# status → mode နဲ့ ဖြည့်တာ
df["status"] = df["status"].fillna(df["status"].mode()[0])

# payment_method → mode နဲ့ ဖြည့်တာ
df["payment_method"] = df["payment_method"].fillna(df["payment_method"].mode()[0])

# region → mode နဲ့ ဖြည့်တာ
df["region"] = df["region"].fillna(df["region"].mode()[0])

# customer_name, gender → "Unknown" ထည့်တာ
df["customer_name"] = df["customer_name"].fillna("Unknown")
df["gender"] = df["gender"].fillna("Unknown")

# စစ်ကြည့်တာ
print("Missing after fix:")
print(df.isnull().sum())

Missing after fix:
order_id          0
customer_id       0
customer_name     0
gender            0
order_date        0
product           0
category          0
quantity          0
unit_price        0
status            0
payment_method    0
region            0
dtype: int64


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True, format='mixed')

# Standard format အဖြစ် သတ်မှတ်
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")

print("Date dtype:", df["order_date"].dtype)
print("Sample dates:", df["order_date"].sample(5).values)

Date dtype: object
Sample dates: ['2023-07-22' '2023-02-02' '2023-12-14' '2023-09-21' '2023-02-10']


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=False, format='mixed')

# စစ်ကြည့်တာ
print(df["order_date"].sample(10).values)


['2023-02-09T00:00:00.000000000' '2023-03-30T00:00:00.000000000'
 '2023-04-11T00:00:00.000000000' '2023-09-12T00:00:00.000000000'
 '2023-07-29T00:00:00.000000000' '2023-04-25T00:00:00.000000000'
 '2023-02-24T00:00:00.000000000' '2023-10-14T00:00:00.000000000'
 '2023-10-19T00:00:00.000000000' '2023-07-10T00:00:00.000000000']


In [ ]:
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")

print("Sample dates:", df["order_date"].sample(5).values)
print("dtype:", df["order_date"].dtype)

Sample dates: ['2023-03-06' '2023-10-01' '2023-02-14' '2023-10-27' '2023-03-14']
dtype: object


In [ ]:
# Cell 12
df["total_price"] = df["quantity"] * df["unit_price"]
print(df[["product","quantity","unit_price","total_price"]].head())

      product  quantity  unit_price  total_price
0    Keyboard         2       85.03       170.06
1  Headphones         5      147.87       739.35
2      Webcam         2       88.85       177.70
3       Mouse         2       43.30        86.60
4       Mouse         3       48.79       146.37


In [ ]:
# Cell 13
df.to_csv("ecommerce_cleaned.csv", index=False)
print("✅ Saved! Final shape:", df.shape)

✅ Saved! Final shape: (500, 13)


In [ ]:
def infer_gender(row):
    if row["gender"] != "Unknown":
        return row["gender"]

    name = str(row["customer_name"]).strip()

    if name.startswith(("Ko ", "U ")):
        return "Male"
    elif name.startswith(("Ma ", "Daw ")):
        return "Female"
    else:
        return "Unknown"

df["gender"] = df.apply(infer_gender, axis=1)

# စစ်ကြည့်တာ
print(df["gender"].value_counts())
print("\nStill Unknown:", (df["gender"] == "Unknown").sum())

gender
Female     252
Male       246
Unknown      2
Name: count, dtype: int64

Still Unknown: 2


In [ ]:
print(df[df["gender"] == "Unknown"][["customer_id", "customer_name", "gender"]])

    customer_id customer_name   gender
83     CUST-288       Unknown  Unknown
396    CUST-136       Unknown  Unknown


In [ ]:
# Unknown 2 ခု ဒီအတိုင်းထားပြီး resave လုပ်တာ
df.to_csv("ecommerce_cleaned.csv", index=False)
print("✅ Final shape:", df.shape)
print(df["gender"].value_counts())

✅ Final shape: (500, 13)
gender
Female     252
Male       246
Unknown      2
Name: count, dtype: int64


In [ ]:
def infer_gender_from_name(row):
    name = str(row["customer_name"]).strip()

    if name.startswith(("Ko ", "U ")):
        return "Male"
    elif name.startswith(("Ma ", "Daw ")):
        return "Female"
    else:
        return "Unknown"  # name က Unknown ဖြစ်နေရင်

df["gender"] = df.apply(infer_gender_from_name, axis=1)

print(df["gender"].value_counts())
print("\nSample check:")
print(df[["customer_name", "gender"]].sample(10))

gender
Male       236
Female     233
Unknown     31
Name: count, dtype: int64

Sample check:
    customer_name  gender
78          U Win    Male
351         U Moe    Male
400        Ma Nwe  Female
353       Ko Htet    Male
336        Daw Su  Female
518        Daw Su  Female
27        Ko Htet    Male
128        Daw Su  Female
505        Daw Su  Female
232        Ma Nwe  Female


In [ ]:
df.to_csv("ecommerce_cleaned.csv", index=False)
print("✅ Saved!")
print("Final shape:", df.shape)
print(df["gender"].value_counts())

✅ Saved!
Final shape: (500, 13)
gender
Male       236
Female     233
Unknown     31
Name: count, dtype: int64


In [ ]:
# CREATE TABLE statement
create_table = """
CREATE TABLE ecommerce (
    order_id VARCHAR(20),
    customer_id VARCHAR(20),
    customer_name VARCHAR(50),
    gender VARCHAR(10),
    order_date DATE,
    product VARCHAR(50),
    category VARCHAR(50),
    quantity INT,
    unit_price DECIMAL(10,2),
    status VARCHAR(20),
    payment_method VARCHAR(20),
    region VARCHAR(20),
    total_price DECIMAL(10,2)
);
"""

rows = []
for _, row in df.iterrows():
    vals = (
        f"'{row['order_id']}'",
        f"'{row['customer_id']}'",
        f"'{row['customer_name']}'",
        f"'{row['gender']}'",
        f"'{row['order_date']}'",
        f"'{row['product']}'",
        f"'{row['category']}'",
        str(int(row['quantity'])),
        str(round(row['unit_price'], 2)),
        f"'{row['status']}'",
        f"'{row['payment_method']}'",
        f"'{row['region']}'",
        str(round(row['total_price'], 2))
    )
    rows.append(f"({', '.join(vals)})")

insert_sql = "INSERT INTO ecommerce VALUES\n" + ",\n".join(rows) + ";"

full_sql = create_table + "\n" + insert_sql

with open("ecommerce_sql.sql", "w") as f:
    f.write(full_sql)

print("✅ SQL file generated!")
print("Total rows:", len(rows))

✅ SQL file generated!
Total rows: 500


In [1]:
# DIM_Customer
dim_customer = df[["customer_id", "customer_name", "gender"]].drop_duplicates()
dim_customer = dim_customer.reset_index(drop=True)
print("DIM_Customer:", dim_customer.shape)
print(dim_customer.head(3))

NameError: name 'df' is not defined

In [2]:
NameError                                 Traceback (most recent call last)
/tmp/ipykernel_1281/3426945232.py in <cell line: 0>()
      1 # DIM_Customer
----> 2 dim_customer = df[["customer_id", "customer_name", "gender"]].drop_duplicates()
      3 dim_customer = dim_customer.reset_index(drop=True)
      4 print("DIM_Customer:", dim_customer.shape)
      5 print(dim_customer.head(3))

NameError: name 'df' is not defined

SyntaxError: invalid decimal literal (3068831151.py, line 2)

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("ecommerce_cleaned.csv")
print("✅ Loaded:", df.shape)
print(df.head(3))

✅ Loaded: (500, 13)
   order_id customer_id customer_name  gender  order_date     product  \
0  ORD-1275    CUST-133        U Kyaw    Male    3/1/2023    Keyboard   
1  ORD-1093    CUST-255       Daw Mya  Female  12/13/2023  Headphones   
2  ORD-1006    CUST-157        Daw Su  Female   7/14/2023      Webcam   

      category  quantity  unit_price     status payment_method region  \
0  Accessories         2       85.03  Completed         Paypal  North   
1        Audio         5      147.87  Completed    Credit Card   East   
2  Accessories         2       88.85  Completed    Credit Card   West   

   total_price  
0       170.06  
1       739.35  
2       177.70  


In [4]:
# DIM_Customer
dim_customer = df[["customer_id", "customer_name", "gender"]].drop_duplicates()
dim_customer = dim_customer.reset_index(drop=True)
print("DIM_Customer:", dim_customer.shape)
print(dim_customer.head(3))

DIM_Customer: (455, 3)
  customer_id customer_name  gender
0    CUST-133        U Kyaw    Male
1    CUST-255       Daw Mya  Female
2    CUST-157        Daw Su  Female


In [5]:
# DIM_Product
dim_product = df[["product", "category"]].drop_duplicates()
dim_product = dim_product.reset_index(drop=True)
dim_product.insert(0, "product_id", range(1, len(dim_product) + 1))
print("DIM_Product:", dim_product.shape)
print(dim_product)

DIM_Product: (8, 3)
   product_id     product     category
0           1    Keyboard  Accessories
1           2  Headphones        Audio
2           3      Webcam  Accessories
3           4       Mouse  Accessories
4           5       Phone  Electronics
5           6     Monitor  Electronics
6           7     USB Hub  Accessories
7           8      Laptop  Electronics


In [6]:
# DIM_Date
df["order_date"] = pd.to_datetime(df["order_date"])

dim_date = df[["order_date"]].drop_duplicates()
dim_date = dim_date.reset_index(drop=True)
dim_date.insert(0, "date_id", range(1, len(dim_date) + 1))
dim_date["month"] = dim_date["order_date"].dt.month
dim_date["quarter"] = dim_date["order_date"].dt.quarter
dim_date["year"] = dim_date["order_date"].dt.year
print("DIM_Date:", dim_date.shape)
print(dim_date.head(5))

DIM_Date: (280, 5)
   date_id order_date  month  quarter  year
0        1 2023-03-01      3        1  2023
1        2 2023-12-13     12        4  2023
2        3 2023-07-14      7        3  2023
3        4 2023-10-21     10        4  2023
4        5 2023-03-06      3        1  2023


In [7]:
# DIM_Location
dim_location = df[["region"]].drop_duplicates()
dim_location = dim_location.reset_index(drop=True)
dim_location.insert(0, "location_id", range(1, len(dim_location) + 1))
print("DIM_Location:", dim_location.shape)
print(dim_location)

# DIM_Payment
dim_payment = df[["payment_method"]].drop_duplicates()
dim_payment = dim_payment.reset_index(drop=True)
dim_payment.insert(0, "payment_id", range(1, len(dim_payment) + 1))
print("\nDIM_Payment:", dim_payment.shape)
print(dim_payment)

DIM_Location: (4, 2)
   location_id region
0            1  North
1            2   East
2            3   West
3            4  South

DIM_Payment: (4, 2)
   payment_id payment_method
0           1         Paypal
1           2    Credit Card
2           3           Cash
3           4     Debit Card


In [8]:
# customer_id တစ်ခုမှာ row ဘယ်နှစ်ခု ရှိလဲ
duplicate_customers = dim_customer.groupby("customer_id").size()
print("Max rows per customer_id:", duplicate_customers.max())
print("customer_id တစ်ခုထက်ပို ရှိတာတွေ:")
print(duplicate_customers[duplicate_customers > 1])

Max rows per customer_id: 7
customer_id တစ်ခုထက်ပို ရှိတာတွေ:
customer_id
CUST-100    4
CUST-101    2
CUST-102    2
CUST-106    2
CUST-108    3
           ..
CUST-293    2
CUST-294    2
CUST-295    3
CUST-296    2
CUST-300    4
Length: 135, dtype: int64


In [9]:
# CUST-100 ကို ဥပမာကြည့်မယ်
print(dim_customer[dim_customer["customer_id"] == "CUST-100"])

    customer_id customer_name   gender
11     CUST-100        Ma Aye   Female
113    CUST-100       Unknown  Unknown
187    CUST-100       Ko Aung     Male
260    CUST-100        U Kyaw     Male


In [10]:
# customer_name + gender ကို တွဲပြီး unique customer တည်ဆောက်တာ
dim_customer = df[["customer_name", "gender"]].drop_duplicates()
dim_customer = dim_customer.reset_index(drop=True)
dim_customer.insert(0, "customer_id", ["CUST-NEW-" + str(i+1).zfill(3) for i in range(len(dim_customer))])

print("DIM_Customer:", dim_customer.shape)
print(dim_customer)

DIM_Customer: (13, 3)
     customer_id customer_name   gender
0   CUST-NEW-001        U Kyaw     Male
1   CUST-NEW-002       Daw Mya   Female
2   CUST-NEW-003        Daw Su   Female
3   CUST-NEW-004       Ko Aung     Male
4   CUST-NEW-005        Ma Nwe   Female
5   CUST-NEW-006        Ma Aye   Female
6   CUST-NEW-007       Ma Thin   Female
7   CUST-NEW-008      Daw Khin   Female
8   CUST-NEW-009        Ko Zaw     Male
9   CUST-NEW-010       Unknown  Unknown
10  CUST-NEW-011         U Win     Male
11  CUST-NEW-012       Ko Htet     Male
12  CUST-NEW-013         U Moe     Male


In [11]:
# Dimension tables တွေနဲ့ merge လုပ်ပြီး FK တွေ ထည့်တာ
fact_orders = df.copy()

# DIM_Customer FK
fact_orders = fact_orders.merge(
    dim_customer[["customer_id", "customer_name", "gender"]],
    on=["customer_name", "gender"],
    how="left"
)

# DIM_Product FK
fact_orders = fact_orders.merge(
    dim_product[["product_id", "product"]],
    on="product",
    how="left"
)

# DIM_Date FK
fact_orders["order_date"] = pd.to_datetime(fact_orders["order_date"])
fact_orders = fact_orders.merge(
    dim_date[["date_id", "order_date"]],
    on="order_date",
    how="left"
)

# DIM_Location FK
fact_orders = fact_orders.merge(
    dim_location[["location_id", "region"]],
    on="region",
    how="left"
)

# DIM_Payment FK
fact_orders = fact_orders.merge(
    dim_payment[["payment_id", "payment_method"]],
    on="payment_method",
    how="left"
)

# FACT table အတွက် လိုတဲ့ column တွေပဲ ထုတ်တာ
fact_orders = fact_orders[[
    "order_id", "customer_id", "product_id",
    "date_id", "location_id", "payment_id",
    "quantity", "unit_price", "total_price", "status"
]]

print("FACT_Orders:", fact_orders.shape)
print(fact_orders.head(3))
print("\nMissing values:")
print(fact_orders.isnull().sum())

KeyError: "['customer_id'] not in index"

In [12]:
fact_orders = df.copy()

# original customer_id ဖယ်တာ (မလိုတော့ဘူး)
fact_orders = fact_orders.drop(columns=["customer_id"])

# DIM_Customer FK
fact_orders = fact_orders.merge(
    dim_customer[["customer_id", "customer_name", "gender"]],
    on=["customer_name", "gender"],
    how="left"
)

# DIM_Product FK
fact_orders = fact_orders.merge(
    dim_product[["product_id", "product"]],
    on="product",
    how="left"
)

# DIM_Date FK
fact_orders["order_date"] = pd.to_datetime(fact_orders["order_date"])
fact_orders = fact_orders.merge(
    dim_date[["date_id", "order_date"]],
    on="order_date",
    how="left"
)

# DIM_Location FK
fact_orders = fact_orders.merge(
    dim_location[["location_id", "region"]],
    on="region",
    how="left"
)

# DIM_Payment FK
fact_orders = fact_orders.merge(
    dim_payment[["payment_id", "payment_method"]],
    on="payment_method",
    how="left"
)

# FACT table အတွက် လိုတဲ့ column တွေပဲ ထုတ်တာ
fact_orders = fact_orders[[
    "order_id", "customer_id", "product_id",
    "date_id", "location_id", "payment_id",
    "quantity", "unit_price", "total_price", "status"
]]

print("FACT_Orders:", fact_orders.shape)
print(fact_orders.head(3))
print("\nMissing values:")
print(fact_orders.isnull().sum())

FACT_Orders: (500, 10)
   order_id   customer_id  product_id  date_id  location_id  payment_id  \
0  ORD-1275  CUST-NEW-001           1        1            1           1   
1  ORD-1093  CUST-NEW-002           2        2            2           2   
2  ORD-1006  CUST-NEW-003           3        3            3           2   

   quantity  unit_price  total_price     status  
0         2       85.03       170.06  Completed  
1         5      147.87       739.35  Completed  
2         2       88.85       177.70  Completed  

Missing values:
order_id       0
customer_id    0
product_id     0
date_id        0
location_id    0
payment_id     0
quantity       0
unit_price     0
total_price    0
status         0
dtype: int64


In [13]:
fact_orders.to_csv("fact_orders.csv", index=False)
dim_customer.to_csv("dim_customer.csv", index=False)
dim_product.to_csv("dim_product.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)
dim_location.to_csv("dim_location.csv", index=False)
dim_payment.to_csv("dim_payment.csv", index=False)

print("✅ All tables saved!")
print(f"FACT_Orders:    {fact_orders.shape}")
print(f"DIM_Customer:   {dim_customer.shape}")
print(f"DIM_Product:    {dim_product.shape}")
print(f"DIM_Date:       {dim_date.shape}")
print(f"DIM_Location:   {dim_location.shape}")
print(f"DIM_Payment:    {dim_payment.shape}")

✅ All tables saved!
FACT_Orders:    (500, 10)
DIM_Customer:   (13, 3)
DIM_Product:    (8, 3)
DIM_Date:       (280, 5)
DIM_Location:   (4, 2)
DIM_Payment:    (4, 2)
